# Gmail Assisstant for Sent Emails

This project connects to your Gmail account and fetches your last 100 sent emails. Those emails are chunked and indexed locally using TF-IDF vectors. When you ask a question, the most relevant chunks are retrieved and passed as context to an LLM via OpenRouter, which then answers based solely on your email history. A Gradio UI provides a simple chat interface to query your emails conversationally.

You'll need the gmailcreds.json from the Google Cloud Console. Get it as follows: <br>

- Go to Google Cloud Console then Enable Gmail API
- Credentials then Create OAuth 2.0 Client ID (Desktop App) and then Download JSON
- Rename the downloaded JSON to gmailcreds.json and place in project root
- Add your Google account as a Test User under OAuth consent screen

On first run, a browser prompt will authorize access and save token.pickle automatically.

In [ ]:
# # If needed, uncomment and run once
# %pip install -q google-auth-oauthlib google-auth-httplib2 google-api-python-client openai scikit-learn python-dotenv

In [ ]:
import os
import re
import json
import base64
import pickle
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from openai import OpenAI

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

load_dotenv(override=True)

PROJECT_DIR = Path.cwd()
CREDENTIALS_FILE = PROJECT_DIR / "gmailcreds.json."
TOKEN_FILE = PROJECT_DIR / "token.pickle"
EMAILS_JSON = PROJECT_DIR / "sent_emails.json"

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_MODEL = "openai/gpt-5.4"

SCOPES = ["https://www.googleapis.com/auth/gmail.readonly"]

if not OPENROUTER_API_KEY:
    raise ValueError("Missing OPENROUTER_API_KEY in environment (.env or system env)")

if not CREDENTIALS_FILE.exists():
    raise FileNotFoundError(
        f"Missing {CREDENTIALS_FILE}. Download OAuth desktop credentials from Google Cloud."
    )

print("Configuration loaded.")

In [ ]:
def _decode_b64(data: str) -> str:
    return base64.urlsafe_b64decode(data).decode("utf-8", errors="ignore")


def _strip_html(html: str) -> str:
    text = re.sub(r"<[^>]+>", " ", html)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def _extract_body(payload: dict[str, Any]) -> str:
    """Extract text/plain if possible, otherwise cleaned text/html."""

    def walk_parts(parts: list[dict[str, Any]]) -> tuple[str, str]:
        plain, html = [], []
        for part in parts:
            mime = part.get("mimeType", "")
            data = part.get("body", {}).get("data", "")

            if mime.startswith("multipart") and "parts" in part:
                nested_plain, nested_html = walk_parts(part["parts"])
                if nested_plain:
                    plain.append(nested_plain)
                if nested_html:
                    html.append(nested_html)
            elif mime == "text/plain" and data:
                plain.append(_decode_b64(data))
            elif mime == "text/html" and data:
                html.append(_strip_html(_decode_b64(data)))

        return "\n".join(plain).strip(), "\n".join(html).strip()

    if "parts" in payload:
        plain_text, html_text = walk_parts(payload["parts"])
        return plain_text if plain_text else html_text

    data = payload.get("body", {}).get("data", "")
    if not data:
        return ""

    decoded = _decode_b64(data)
    return _strip_html(decoded) if payload.get("mimeType") == "text/html" else decoded.strip()


def _header_map(headers: list[dict[str, str]]) -> dict[str, str]:
    return {h.get("name", "").lower(): h.get("value", "") for h in headers}


@dataclass
class GmailSentEmailFetcher:
    credentials_file: Path
    token_file: Path
    scopes: list[str]

    def authenticate(self):
        creds = None

        if self.token_file.exists():
            with open(self.token_file, "rb") as f:
                creds = pickle.load(f)

        if not creds or not creds.valid:
            if creds and creds.expired and creds.refresh_token:
                creds.refresh(Request())
            else:
                flow = InstalledAppFlow.from_client_secrets_file(
                    str(self.credentials_file), self.scopes
                )
                creds = flow.run_local_server(port=0)

            with open(self.token_file, "wb") as f:
                pickle.dump(creds, f)

        return build("gmail", "v1", credentials=creds)

    def fetch_sent_emails(self, max_results: int = 100) -> list[dict[str, str]]:
        service = self.authenticate()

        try:
            result = (
                service.users()
                .messages()
                .list(userId="me", q="label:SENT", maxResults=max_results)
                .execute()
            )
            message_refs = result.get("messages", [])
        except HttpError as err:
            raise RuntimeError(f"Failed to list messages: {err}") from err

        emails: list[dict[str, str]] = []

        for ref in message_refs:
            msg = (
                service.users()
                .messages()
                .get(userId="me", id=ref["id"], format="full")
                .execute()
            )

            headers = _header_map(msg.get("payload", {}).get("headers", []))
            body = _extract_body(msg.get("payload", {}))

            emails.append(
                {
                    "id": msg.get("id", ""),
                    "subject": headers.get("subject", "No Subject"),
                    "from": headers.get("from", "Unknown"),
                    "to": headers.get("to", "Unknown"),
                    "date": headers.get("date", "Unknown"),
                    "body": body,
                }
            )

        return emails

In [ ]:
def save_emails(path: Path, emails: list[dict[str, str]]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(emails, f, indent=2, ensure_ascii=False)


def load_emails(path: Path) -> list[dict[str, str]]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


# Run this cell to fetch fresh sent emails from Gmail.
fetcher = GmailSentEmailFetcher(
    credentials_file=CREDENTIALS_FILE,
    token_file=TOKEN_FILE,
    scopes=SCOPES,
)

emails = fetcher.fetch_sent_emails(max_results=100)
save_emails(EMAILS_JSON, emails)

print(f"Saved {len(emails)} sent emails to {EMAILS_JSON}")
print("Sample subject:", emails[0]["subject"] if emails else "No emails found")

In [ ]:
def chunk_text(text: str, chunk_size: int = 700, overlap: int = 120) -> list[str]:
    text = (text or "").strip()
    if not text:
        return []

    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start = max(end - overlap, 0)

    return chunks


def email_to_document(email: dict[str, str]) -> str:
    return (
        f"Date: {email.get('date', 'Unknown')}\n"
        f"From: {email.get('from', 'Unknown')}\n"
        f"To: {email.get('to', 'Unknown')}\n"
        f"Subject: {email.get('subject', 'No Subject')}\n\n"
        f"{email.get('body', '')}"
    ).strip()


class LocalRAGIndex:
    """
    Efficient local retrieval index using TF-IDF.
    Faster setup than embedding every chunk externally and good for personal notes/email QA.
    """

    def __init__(self):
        self.vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)
        self.chunk_texts: list[str] = []
        self.chunk_meta: list[dict[str, str]] = []
        self.matrix = None

    def build(self, emails: list[dict[str, str]], chunk_size: int = 700, overlap: int = 120):
        texts: list[str] = []
        metas: list[dict[str, str]] = []

        for email in emails:
            doc = email_to_document(email)
            for i, piece in enumerate(chunk_text(doc, chunk_size=chunk_size, overlap=overlap)):
                texts.append(piece)
                metas.append(
                    {
                        "email_id": email.get("id", ""),
                        "subject": email.get("subject", ""),
                        "to": email.get("to", ""),
                        "date": email.get("date", ""),
                        "chunk": str(i),
                    }
                )

        if not texts:
            raise ValueError("No text chunks available. Fetch emails first.")

        self.chunk_texts = texts
        self.chunk_meta = metas
        self.matrix = self.vectorizer.fit_transform(texts)

    def retrieve(self, query: str, k: int = 8) -> list[dict[str, Any]]:
        if self.matrix is None:
            raise RuntimeError("Index not built. Call build() first.")

        qvec = self.vectorizer.transform([query])
        scores = cosine_similarity(qvec, self.matrix).flatten()
        top_indices = scores.argsort()[::-1][:k]

        return [
            {
                "score": float(scores[i]),
                "text": self.chunk_texts[i],
                "meta": self.chunk_meta[i],
            }
            for i in top_indices
            if scores[i] > 0
        ]


emails_for_rag = load_emails(EMAILS_JSON)
rag_index = LocalRAGIndex()
rag_index.build(emails_for_rag)

print(f"RAG index ready with {len(rag_index.chunk_texts)} chunks.")

In [ ]:
client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
)

SYSTEM_PROMPT = (
    "You are a personal email assistant. "
    "Answer using only the provided email context. "
    "If context is insufficient, say you do not have enough evidence in sent emails. "
    "Be concise and include who/when when possible."
)


def build_context(hits: list[dict[str, Any]]) -> str:
    blocks = []
    for i, h in enumerate(hits, start=1):
        m = h["meta"]
        blocks.append(
            f"[Chunk {i}] score={h['score']:.3f}\n"
            f"Subject: {m.get('subject', '')}\n"
            f"To: {m.get('to', '')}\n"
            f"Date: {m.get('date', '')}\n"
            f"Text:\n{h['text']}"
        )
    return "\n\n".join(blocks)


def answer_with_rag(question: str, history: list[dict[str, str]] | None = None, k: int = 8) -> str:
    history = history or []

    # Query expansion from prior user turns improves continuity without sending entire email corpus.
    prior_user = "\n".join(m["content"] for m in history if m.get("role") == "user")
    retrieval_query = f"{prior_user}\n{question}".strip()

    hits = rag_index.retrieve(retrieval_query, k=k)
    context = build_context(hits) if hits else "No relevant chunks found."

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": f"Context from sent emails:\n\n{context}\n\nQuestion: {question}",
        },
    ]

    resp = client.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=messages,
        temperature=0.1,
    )
    return resp.choices[0].message.content


# Quick test
test_q = "Who did I email recently about automation?"
print(answer_with_rag(test_q)[:1000])

In [ ]:
chat_history: list[dict[str, str]] = []


def ask(question: str) -> str:
    answer = answer_with_rag(question, history=chat_history, k=8)
    chat_history.append({"role": "user", "content": question})
    chat_history.append({"role": "assistant", "content": answer})
    return answer


# Example:
# print(ask("What did I say about an investment agreement"))

In [ ]:
import gradio as gr


def respond(user_message: str, history: list[dict[str, str]]):
    """Custom Gradio handler using existing RAG pipeline."""
    history = history or []
    answer = answer_with_rag(user_message, history=history, k=8)

    updated_history = history + [
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": answer},
    ]
    return updated_history, updated_history


def clear_chat():
    return [], []


with gr.Blocks(title="Email Assistant") as demo:
    gr.Markdown(
        "## Gmail Assistant\n"
        "Ask questions about your sent Gmail emails using local retrieval + OpenRouter."
    )

    chatbot = gr.Chatbot(type="messages", height=420)
    state = gr.State([])

    with gr.Row():
        msg = gr.Textbox(
            label="Question",
            placeholder="e.g., Who did I email about project deadlines?",
            lines=2,
            scale=8,
        )
        send_btn = gr.Button("Send", variant="primary", scale=1)

    with gr.Row():
        clear_btn = gr.Button("Clear")

    examples = gr.Examples(
        examples=[
            "Who have I emailed about meetings?",
            "What projects have I discussed recently?",
            "Find emails where I mentioned invoices",
        ],
        inputs=msg,
    )

    send_btn.click(respond, inputs=[msg, state], outputs=[chatbot, state])
    msg.submit(respond, inputs=[msg, state], outputs=[chatbot, state])
    clear_btn.click(clear_chat, inputs=None, outputs=[chatbot, state])



demo.launch()